# Equity options (Black–76 + Monte Carlo benchmark)

**Start here:** This deep dive expands on `02_pricing/pricing_across_asset_classes.ipynb`; use `02_pricing/pricing_fundamentals.ipynb` for the shared instrument JSON -> market -> model pipeline.

**Prerequisites:** `02_pricing/pricing_across_asset_classes.ipynb`. **`equity_option`** with **`black76`**; compare to **`finstack_quant.monte_carlo`** European pricer as an independent simulation check.


## Concept

- **Black–76** on forwards: discounting, dividend yield, vol surface.
- **Delta** (and other Greeks) come from the analytical pricer when registered.
- **Monte Carlo** reproduces the same risk-neutral expectation subject to simulation noise.


In [1]:
import json
from datetime import date

import sys
sys.path.insert(0, "../..")

from _shared import DEMO_AS_OF, print_metrics, usd_ois_curve

from finstack_quant.valuations import (
    ValuationResult,
    bs_greeks,
    bs_implied_vol,
    black76_implied_vol,
    lookback_option_price,
)
from finstack_quant.valuations.instruments import price_instrument_with_metrics, validate_instrument_json
from finstack_quant.core.market_data import MarketContext, VolSurface
from finstack_quant.monte_carlo import EuropeanPricer

print("Imports OK (equity options).")


Imports OK (equity options).


## Instrument JSON


In [2]:
AS_OF = DEMO_AS_OF
AS_OF_STR = AS_OF.isoformat()

equity_option = json.dumps(
    {
        "type": "equity_option",
        "spec": {
            "id": "SPX-CALL-4500",
            "underlying_ticker": "SPX",
            "strike": 4500.0,
            "option_type": "call",
            "exercise_style": "european",
            "expiry": "2026-06-15",
            "notional": {"amount": 100.0, "currency": "USD"},
            "day_count": "Act365F",
            "settlement": "cash",
            "discount_curve_id": "USD-OIS",
            "spot_id": "EQUITY-SPOT",
            "vol_surface_id": "EQUITY-VOL",
            "div_yield_id": "EQUITY-DIVYIELD",
            "pricing_overrides": {},
            "attributes": {},
        },
    }
)

validate_instrument_json(equity_option)
print("equity_option JSON OK")


equity_option JSON OK


## MarketContext


In [3]:
ois = usd_ois_curve(AS_OF)
mc = MarketContext().insert(ois)
exp_eq = [0.25, 0.5, 1.0, 2.0]
strikes_eq = [4000.0, 4300.0, 4500.0, 4700.0, 5000.0]
mc.insert(VolSurface("EQUITY-VOL", exp_eq, strikes_eq, [0.22] * (len(exp_eq) * len(strikes_eq))))
mc.insert_price("EQUITY-SPOT", 4500.0, "USD")
mc.insert_price("EQUITY-DIVYIELD", 0.015)
market_json = mc.to_json()
print("surface:", mc.get_surface("EQUITY-VOL").id)

surface: EQUITY-VOL


## Pricing


In [4]:
out = price_instrument_with_metrics(
    equity_option, market_json, AS_OF_STR, model="black76", metrics=["delta"]
)
vr = ValuationResult.from_json(out)
print("JSON black76:", vr)
print_metrics(vr, ["delta"])


JSON black76: ValuationResult(id="SPX-CALL-4500", price=50511.4209, currency=USD, metrics=1)
  delta: 57.43025031192922


## Metrics & MC cross-check


In [5]:
# Independent GBM Monte Carlo on a stylized underlying (not the JSON pricer MC path).
spot = 4500.0
strike = 4500.0
q = 0.015
vol = 0.22
expiry = date(2026, 6, 15)
T = (expiry - AS_OF).days / 365.0

# Discount off the SAME curve object the JSON pricer sees: DiscountCurve.df(t) applies the
# Rust interpolation convention, and .zero(t) returns the continuously-compounded zero rate.
df_T = ois.df(T)
r = ois.zero(T)

pricer = EuropeanPricer(num_paths=40_000, seed=42)
mc_res = pricer.price_call(spot=spot, strike=strike, rate=r, div_yield=q, vol=vol, expiry=T)
print("MC call (per unit notional scale): mean=", mc_res.mean.amount, "stderr=", mc_res.stderr)
print(
    "Analytical black76 from JSON above; MC uses the USD-OIS zero rate r=",
    round(r, 6),
    "DF(T)=",
    round(df_T, 6),
    "T=",
    round(T, 4),
)


MC call (per unit notional scale): mean= 506.188241343781 stderr= 4.068502730455617
Analytical black76 from JSON above; MC uses the USD-OIS zero rate r= 0.031291 DF(T)= 0.956728 T= 1.4137


## Implied volatility override (flat σ across tenor and strike)

`pricing_overrides.implied_volatility` short-circuits the vol surface and uses a **flat σ** for pricing. It is honored uniformly across vanilla options (Black-76) and all surface-driven Monte Carlo pricers: **autocallables, cliquets, Asian, lookback, barrier, quanto, FX barrier** — anywhere the pricer previously read `get_surface(id).value_clamped(t, K)`.

This is the canonical lever for:

- **Revaluation workflows** — quote the bond/option at a specified implied vol without hand-editing a surface snapshot.
- **Vega scenarios** — shift an instrument to a flat σ level for risk reports.
- **Reproducing client-quoted prices** — use the broker's implied vol directly.

In [6]:
def price_option_at_vol(implied_vol: float | None) -> ValuationResult:
    """Reprice the SPX call with an optional flat-σ override."""
    spec = json.loads(equity_option)
    if implied_vol is not None:
        spec["spec"]["pricing_overrides"] = {"implied_volatility": implied_vol}
    out = price_instrument_with_metrics(
        json.dumps(spec), market_json, AS_OF_STR, model="black76", metrics=["delta"]
    )
    return ValuationResult.from_json(out)


# Surface σ across the grid is 0.22; override to 0.30 (fatter) and 0.15 (tighter).
for iv in (None, 0.15, 0.22, 0.30):
    vr = price_option_at_vol(iv)
    tag = "surface" if iv is None else f"flat σ = {iv:.2f}"
    print(f"{tag:<18s} price = {vr.price:>10,.4f} USD  delta = {vr.get_metric('delta'):.4f}")

surface            price = 50,511.4209 USD  delta = 57.4303
flat σ = 0.15      price = 36,220.4536 USD  delta = 57.4102
flat σ = 0.22      price = 50,511.4209 USD  delta = 57.4303
flat σ = 0.30      price = 66,793.7914 USD  delta = 58.3460


## Analytical Greeks & implied vol (closed-form helpers)

Beyond the JSON pricing path, `finstack_quant.valuations` exposes closed-form analytics: `bs_greeks` (full Greek set), `bs_implied_vol` / `black76_implied_vol` (vol inversion), and `lookback_option_price` (continuous-monitoring lookback).

In [7]:
greeks = bs_greeks(spot=100.0, strike=100.0, r=0.05, q=0.0, sigma=0.20, t=1.0, is_call=True)
print("bs_greeks:", {k: round(v, 4) for k, v in greeks.items()})

# Invert price -> implied vol (round-trips the 20% input)
iv = bs_implied_vol(spot=100.0, strike=100.0, r=0.05, q=0.0, t=1.0, price=10.4506, is_call=True)
print("bs_implied_vol:", round(iv, 4))

iv76 = black76_implied_vol(forward=100.0, strike=100.0, df=0.95, t=1.0, price=8.0, is_call=True)
print("black76_implied_vol:", round(iv76, 4))

# Fixed-strike lookback call given the realized minimum extremum
lb = lookback_option_price(100.0, 100.0, 0.05, 0.0, 0.20, 1.0, extremum=95.0, strike_type="fixed", is_call=True)
print("lookback_option_price:", round(lb, 4))

bs_greeks: {'delta': 0.6368, 'gamma': 0.0188, 'vega': 0.3752, 'theta': -0.0176, 'rho': 0.5323, 'rho_q': -0.6368}
bs_implied_vol: 0.2
black76_implied_vol: 0.2115
lookback_option_price: 17.2168


## Takeaways

- **`black76`** is the JSON model key for vanilla equity options here.
- **`finstack_quant.monte_carlo.EuropeanPricer`** gives a simulation benchmark; align vol, div yield, rate, and horizon when comparing. Read the rate straight off the curve object with **`DiscountCurve.zero(t)`** (and `DiscountCurve.df(t)`) so the benchmark uses the same interpolation convention as the JSON pricer.
- JSON **`price_instrument_with_metrics`** does not currently register `monte_carlo_gbm` for `equity_option` in this build.
- **`implied_volatility` override** now applies uniformly across vanilla and path-dependent MC pricers as a flat σ — no more per-pricer ad-hoc handling.